# BFCL V4 — P06 smoke test (official harness)

Purpose: **actually run**, in a user-controlled GPU Colab session, one indexed family across base / missing function / missing parameter with a supported 4B model. This is a **smoke test**, not scientific evidence for a published result. This notebook was authored but **has not been run**. GPU must be enabled in Colab. Notebook pins the official Gorilla source commit; model revision is reported from resolved Hugging Face download information and should be frozen for full experiments. Data and model may require a substantial download.

2026-10-29 troubleshooting revision: BFCL runs in an isolated Python 3.12 environment, with streamed setup/inference/evaluation logs. This revision has not been executed on Colab. If a cell fails, download the corresponding file from /content/P06_BFCL_Logs using Colab's Files panel. Do not continue to generation after setup failure.

Dependency repair: explicitly install soundfile in the Python 3.12 environment and test the complete BFCL CLI import chain before inference. User rerun reached CLI startup but failed before model inference; successful GPU execution remains unverified.

Tokenizer/server revision: pin Transformers 4.51.3 and validate the Qwen tokenizer through vLLM before generation. Launch vLLM separately using the official BFCL skip-server-setup option, choose FP16 on pre-Ampere GPUs, limit server context to 8192 and concurrency to one, and record the full serving profile. This resource-limited smoke test is not equivalent to a full-context leaderboard run. Context overflow must be reported as a runtime limitation. Model execution of this revision remains unverified.


In [ ]:
import subprocess, sys, os, json, pathlib, datetime, platform
print('UTC', datetime.datetime.now(datetime.timezone.utc).isoformat())
print('Python', sys.version, 'Platform',platform.platform())
subprocess.run(['nvidia-smi'],check=True)



In [ ]:
from pathlib import Path
root=Path('/content/gorilla')
if not root.exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/ShishirPatil/gorilla.git',str(root)],check=True)
commit='6ea57973c7a6097fd7c5915698c54c17c5b1b6c8'
subprocess.run(['git','-C',str(root),'checkout',commit],check=True)
work=root/'berkeley-function-call-leaderboard'
print('Commit:',subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip())
# BFCL at this commit pins numpy==1.26.4 and vllm==0.8.5.
# Use an isolated interpreter; the Colab notebook kernel may remain Python 3.13.
log_dir=Path('/content/P06_BFCL_Logs')
log_dir.mkdir(exist_ok=True)
def run_logged(cmd, name, cwd=None, env=None):
    log_path=log_dir/(name+'.log')
    print('Running:', ' '.join(map(str,cmd)), flush=True)
    with log_path.open('w') as out:
        proc=subprocess.Popen(list(map(str,cmd)), cwd=cwd, env=env,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            out.write(line)
            out.flush()
        rc=proc.wait()
    if rc:
        print('Saved failure log:', log_path)
        raise subprocess.CalledProcessError(rc, cmd)
    return log_path

run_logged([sys.executable,'-m','pip','install','uv'], 'install_uv')
uv_cmd=[sys.executable,'-m','uv']
run_logged(uv_cmd+['python','install','3.12'], 'install_python312')
venv=Path('/content/bfcl-py312')
if not (venv/'bin/python').exists():
    run_logged(uv_cmd+['venv','--python','3.12','--seed',str(venv)], 'create_venv')
bfcl_python=venv/'bin/python'
bfcl_cli=venv/'bin/bfcl'
run_logged([bfcl_python,'-c','import sys; print(sys.version); assert sys.version_info[:2] == (3,12)'], 'verify_python')
# vLLM 0.8.5 uses tokenizer APIs present in Transformers 4.51.3.
# Apply the constraint during resolution so a future major version is not selected.
constraints=Path('/content/P06_BFCL_constraints.txt')
constraints.write_text('transformers==4.51.3\n')
run_logged([bfcl_python,'-m','pip','install','-c',str(constraints),'-e',str(work)+'[oss_eval_vllm]'], 'install_bfcl')
# Qwen-Agent imports soundfile at CLI startup, including for text-only runs.
run_logged([bfcl_python,'-m','pip','install','soundfile'], 'install_soundfile')
run_logged([bfcl_python,'-m','pip','check'], 'pip_check')
run_logged([bfcl_python,'-c','import numpy, torch, vllm; print("numpy", numpy.__version__); print("torch", torch.__version__); print("vllm", vllm.__version__); print("CUDA", torch.cuda.is_available())'], 'verify_imports')
# Exercise the complete CLI import chain before attempting GPU inference.
preflight_env=os.environ.copy()
preflight_env['PATH']=str(venv/'bin')+os.pathsep+preflight_env.get('PATH','')
preflight_env['BFCL_PROJECT_ROOT']=str(work)
run_logged([bfcl_python,'-c','import soundfile; from qwen_agent.llm import get_chat_model; import bfcl_eval.__main__; print("BFCL/Qwen-Agent imports OK; soundfile", soundfile.__version__)'], 'verify_cli_imports', cwd=work, env=preflight_env)
run_logged([bfcl_cli,'generate','--help'], 'verify_generate_cli', cwd=work, env=preflight_env)
run_logged([bfcl_cli,'evaluate','--help'], 'verify_evaluate_cli', cwd=work, env=preflight_env)
# Download tokenizer/config only, then exercise vLLM's actual tokenizer wrapper.
tokenizer_check = """
import json, torch, transformers
from transformers import AutoTokenizer, AutoConfig
from vllm.transformers_utils.tokenizer import get_cached_tokenizer
model_id='Qwen/Qwen3-4B-Instruct-2507'
assert transformers.__version__ == '4.51.3', transformers.__version__
tok=AutoTokenizer.from_pretrained(model_id)
assert hasattr(tok, 'all_special_tokens_extended')
cached=get_cached_tokenizer(tok)
assert cached.encode('BFCL smoke test') == tok.encode('BFCL smoke test')
cfg=AutoConfig.from_pretrained(model_id)
print(json.dumps({'transformers':transformers.__version__, 'tokenizer':type(tok).__name__,
 'model_revision':getattr(cfg,'_commit_hash',None), 'gpu':torch.cuda.get_device_name(0),
 'capability':torch.cuda.get_device_capability(0), 'tokenizer_check':'PASS'}))
"""
run_logged([bfcl_python,'-c',tokenizer_check], 'verify_tokenizer', cwd=work, env=preflight_env)
run_logged([bfcl_python,'-m','pip','freeze'], 'pip_freeze')



In [ ]:
config={'multi_turn_base':['multi_turn_base_1'], 'multi_turn_miss_func':['multi_turn_miss_func_1'],'multi_turn_miss_param':['multi_turn_miss_param_1']}
(work/'test_case_ids_to_generate.json').write_text(json.dumps(config,indent=2))
print('Selected smoke-test IDs:',config)
print('Do not extrapolate results from one family.')



In [ ]:
# Start a resource-limited server via BFCL's official --skip-server-setup path.
# This avoids modifying BFCL source, prompts, decoding or the evaluator.
import signal, socket, time, urllib.request
model='Qwen/Qwen3-4B-Instruct-2507-FC'
model_checkpoint='Qwen/Qwen3-4B-Instruct-2507'
os.environ['BFCL_PROJECT_ROOT']=str(work)
run_env=os.environ.copy()
run_env['PATH']=str(venv/'bin')+os.pathsep+run_env.get('PATH','')
# Remove stale remote endpoint overrides from earlier notebook experiments.
for key in ('REMOTE_OPENAI_BASE_URL','REMOTE_OPENAI_TOKENIZER_PATH','REMOTE_OPENAI_API_KEY'):
    run_env.pop(key, None)
run_env['LOCAL_SERVER_ENDPOINT']='127.0.0.1'
# Select a free local port rather than reuse a possibly occupied server.
with socket.socket() as sock:
    sock.bind(('127.0.0.1',0))
    port=sock.getsockname()[1]
run_env['LOCAL_SERVER_PORT']=str(port)
run_env['VLLM_USE_V1']='0'
run_env['PYTHONUNBUFFERED']='1'
capability=json.loads(subprocess.check_output(
    [str(bfcl_python),'-c','import json, torch; print(json.dumps(list(torch.cuda.get_device_capability(0))))'],
    text=True))
server_dtype='half' if capability[0] < 8 else 'bfloat16'
server_max_len=8192  # Smoke-test resource cap; not the model's native 262144.
server_cmd=[str(venv/'bin/vllm'),'serve',model_checkpoint,
    '--host','127.0.0.1','--port',str(port),'--dtype',server_dtype,
    '--tensor-parallel-size','1','--gpu-memory-utilization','0.85',
    '--max-model-len',str(server_max_len),'--max-num-seqs','1',
    '--enforce-eager','--seed','0']
cmd=[str(bfcl_cli),'generate','--model',model,'--run-ids',
     '--backend','vllm','--skip-server-setup','--num-threads','1','--include-input-log']
profile={'server_command':server_cmd,'generation_command':cmd,
         'dtype':server_dtype,'context_cap':server_max_len,'gpu_capability':capability,
         'note':'Resource-limited smoke profile. Overlong requests must be recorded as runtime errors, never silently truncated or treated as model failures.'}
(log_dir/'runtime_profile.json').write_text(json.dumps(profile,indent=2))
server_log=log_dir/'vllm_server.log'
print('Server command:', ' '.join(server_cmd), flush=True)
print('Server log:',server_log, flush=True)
server=None
try:
    with server_log.open('w') as out:
        server=subprocess.Popen(server_cmd,stdout=out,stderr=subprocess.STDOUT,
                                env=run_env,cwd=work,start_new_session=True)
        deadline=time.monotonic()+1800
        next_update=0
        while True:
            if server.poll() is not None:
                print(server_log.read_text(errors='replace')[-16000:])
                raise RuntimeError(f'vLLM exited with {server.returncode}; see {server_log}')
            try:
                with urllib.request.urlopen(f'http://127.0.0.1:{port}/v1/models',timeout=2) as response:
                    served=json.load(response)
                if model_checkpoint not in [item['id'] for item in served['data']]:
                    raise RuntimeError('Unexpected model served on selected port')
                print('Server ready.',flush=True)
                break
            except (OSError, ValueError):
                pass
            now=time.monotonic()
            if now>=deadline:
                print(server_log.read_text(errors='replace')[-16000:])
                raise TimeoutError(f'Server not ready after 30 minutes; see {server_log}')
            if now>=next_update:
                print('Waiting for model server; downloading/loading may take several minutes.',flush=True)
                next_update=now+30
            time.sleep(2)
        run_logged(cmd,'generate',cwd=work,env=run_env)
finally:
    # Stop only the process group created by this cell, including worker children.
    if server is not None:
        try:
            os.killpg(server.pid,signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            pass
        try:
            os.killpg(server.pid,signal.SIGKILL)
        except ProcessLookupError:
            pass
        server.wait()


In [ ]:
categories='multi_turn_base,multi_turn_miss_func,multi_turn_miss_param'
cmd=[str(bfcl_cli),'evaluate','--model',model,'--test-category',categories,'--partial-eval']
run_env=os.environ.copy()
run_env['PATH']=str(venv/'bin')+os.pathsep+run_env.get('PATH','')
run_logged(cmd, 'evaluate', cwd=work, env=run_env)



In [ ]:
from google.colab import files
import shutil, hashlib
bundle=Path('/content/P06_BFCL_Smoke_Outputs')
bundle.mkdir(exist_ok=True)
if log_dir.exists():
    shutil.copytree(log_dir,bundle/'logs',dirs_exist_ok=True)
for sub in ('result','score'):
    src=work/sub
    if src.exists():
        for path in src.rglob('*'):
            if path.is_file():
                dest=bundle/sub/path.relative_to(src)
                dest.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(path,dest)
manifest={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'bfcl_commit':commit,'model_id':model,'selected_ids':config,'note':'Smoke test, not complete study','files':{}}
for p in bundle.rglob('*'):
    if p.is_file():manifest['files'][str(p.relative_to(bundle))]=hashlib.sha256(p.read_bytes()).hexdigest()
(bundle/'manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive('/content/P06_BFCL_Smoke_Outputs','zip',bundle)
print('Artifacts:',archive,'files:',len(manifest['files']))
files.download(archive)

